# Importaciones

In [37]:
from pathlib import Path
import os

import pandas as pd
import duckdb
import psycopg

from dotenv import load_dotenv

load_dotenv()

True

In [13]:
DATABASE_URL = os.getenv("POSTGRES_URI")

In [14]:
# os.environ.pop('POSTGRES_URI', None)

# Prueba de conexión a base de datos

In [34]:
try:
    with psycopg.connect(conninfo=DATABASE_URL) as conn:
        print("✅ Conexión exitosa con psycopg3!")
        
        with conn.cursor() as cur:
            print("\nConsultando la tabla 'news_chile'...")
            cur.execute("SELECT * FROM news_chile LIMIT 1;")
            
            records = cur.fetchall()
            
            if not records:
                print("  La tabla 'news' está vacía.")
            else:
                for row in records:
                    print(f"  - {row}")

except psycopg.OperationalError as e:
    print(f"❌ Error de conexión: {e}\nPor favor, verifica la IP del Asus y que el Firewall permita el tráfico.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

✅ Conexión exitosa con psycopg3!

Consultando la tabla 'news_chile'...
  - (158443, 'www.aricaldia.cl/tarifas-servel-2024', 'https://www.aricaldia.cl/alcalde-cristian-zavala-llama-a-construir-un-estado-inclusivo-con-las-comunas-rurales/', datetime.datetime(2024, 12, 7, 13, 24, 51), None, 'En la jornada de asunción de nuevas autoridades de la comuna, el alcalde de Camarones, Cristian Zavala Soto, asumió un nuevo período al frente del gobierno comunal con un llamado contundente: construir un Estado que incluya y valore el territorio rural.\n\nDurante la ceremonia de asunción, marcada por la diversidad cultural y espiritual, Zavala expresó la necesidad de que las decisiones del Estado comprendan las complejidades de las zonas rurales y se adapten a las características únicas de cada territorio.\n\n“La ruralidad no puede estar al margen de las políticas públicas; al contrario, debe ser el centro de las decisiones para cerrar las brechas de desigualdad. Cada rincón del territorio debe ser p

# Consultas de exploración

In [35]:
with psycopg.connect(conninfo=DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(body_hash) AS raw_count, count(DISTINCT body_hash) as deduplicated_count FROM news_chile LIMIT 2;")
        
        records = cur.fetchall()
        
        if not records:
            print("  La tabla 'news' está vacía.")
        else:
            for row in records:
                print(f"  - {row}")

  - (333762, 333762)


In [36]:
with psycopg.connect(conninfo=DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT date FROM news_chile LIMIT 10;")
        
        records = cur.fetchall()
        
        if not records:
            print("  La tabla 'news' está vacía.")
        else:
            for row in records:
                print(f"  - {row}")

  - (datetime.datetime(2024, 12, 7, 13, 24, 51),)
  - (datetime.datetime(2024, 12, 7, 2, 37, 43),)
  - (datetime.datetime(2024, 12, 7, 2, 19, 43),)
  - (datetime.datetime(2024, 12, 8, 0, 42, 2),)
  - (datetime.datetime(2024, 12, 7, 22, 1),)
  - (datetime.datetime(2024, 12, 8, 0, 52, 21),)
  - (datetime.datetime(2024, 12, 8, 0, 29, 13),)
  - (datetime.datetime(2024, 12, 7, 13, 12, 3),)
  - (datetime.datetime(2024, 12, 7, 0, 0),)
  - (datetime.datetime(2024, 12, 7, 0, 0),)


In [33]:
with psycopg.connect(conninfo=DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT EXTRACT(MONTH FROM date) as month, count(body_hash) FROM news_chile GROUP BY month;")
        
        records = cur.fetchall()
        
        if not records:
            print("  La tabla 'news' está vacía.")
        else:
            for row in records:
                print(f"  - {row}")

  - (None, 38447)
  - (Decimal('11'), 17880)
  - (Decimal('7'), 47630)
  - (Decimal('8'), 43324)
  - (Decimal('9'), 32812)
  - (Decimal('6'), 33974)
  - (Decimal('1'), 15082)
  - (Decimal('2'), 12483)
  - (Decimal('3'), 16545)
  - (Decimal('10'), 14202)
  - (Decimal('5'), 29528)
  - (Decimal('12'), 14516)
  - (Decimal('4'), 17339)


In [38]:
con = duckdb.connect()
con.sql("INSTALL postgres;")
con.sql("LOAD postgres;")

db_uri = DATABASE_URL

# Adjuntar usando f-string para insertar la variable
con.sql(f"ATTACH '{db_uri}' AS news_db (TYPE postgres, READ_ONLY);")

# Realizar la consulta
resultado = con.sql("SELECT * FROM news_db.news_chile LIMIT 10").df()
print(resultado)

       id                                   media_name  \
0  158443         www.aricaldia.cl/tarifas-servel-2024   
1  158444         www.aricaldia.cl/tarifas-servel-2024   
2  158445         www.aricaldia.cl/tarifas-servel-2024   
3  158446                   elsoldeiquique.cl/nacional   
4  158447                   elsoldeiquique.cl/nacional   
5  158448                   elsoldeiquique.cl/nacional   
6  158449                   elsoldeiquique.cl/nacional   
7  158450                    www.diarioantofagasta.cl/   
8  158451  www.antofagastanoticias.cl/category/turismo   
9  158452                           elamerica.cl/feed/   

                                                 url                date  \
0  https://www.aricaldia.cl/alcalde-cristian-zava... 2024-12-07 13:24:51   
1  https://www.aricaldia.cl/160-anos-que-dan-cuen... 2024-12-07 02:37:43   
2  https://www.aricaldia.cl/embellecen-centro-de-... 2024-12-07 02:19:43   
3  https://elsoldeiquique.cl/tarapaca-camaras-tra... 2024